In [8]:
import re
import nltk

In [11]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

/home/thiago/Diplotokens-P1/venv/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/thiago/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


True

In [ ]:
import re
import nltk

PRESSURE_UNIT = r"mmHg"
WEIGHT_UNIT = r"(?:kg|mg|mcg|ng|hg|g)"
VOLUME_UNIT = r"(?:ml|dl|l)"
TIME_UNIT_EXT = r"(?:min|hr|hrs|h|s|sec|m|days?|weeks?|months?|years?|y/o|yo|-years-old)"
LENGTH_UNIT = r"(?:cm|mm)"
SPECIAL_UNIT = r"(?:iu|U|mmol)"

TOKEN_SEP_REGEX = re.compile(
    rf"""
        \d+(?:,\d{{3}})*(?:\.\d+)?\s?{LENGTH_UNIT}\s*[xX]\s*\d+(?:,\d{{3}})*(?:\.\d+)?\s?{LENGTH_UNIT}(?:\s*[xX]\s*\d+(?:,\d{{3}})*(?:\.\d+)?\s?{LENGTH_UNIT})?(?![A-Za-z])
        | \d+(?:,\d{{3}})*(?:\.\d+)?
        (?:
            /\d+(?:,\d{{3}})*(?:\.\d+)?(?:\s?{WEIGHT_UNIT}|\s?{VOLUME_UNIT})?(?![A-Za-z])
            | /{TIME_UNIT_EXT}(?![A-Za-z])
            | \s?(?:{PRESSURE_UNIT}|{WEIGHT_UNIT}|{VOLUME_UNIT}|{TIME_UNIT_EXT}|{LENGTH_UNIT}|{SPECIAL_UNIT})(?:/(?:{WEIGHT_UNIT}|{VOLUME_UNIT}|{TIME_UNIT_EXT}))?(?![A-Za-z])
        )?
        | [A-Za-z]+(?:-[A-Za-z]+)*
        | [.,;:()!?]+
    """,
    re.VERBOSE | re.IGNORECASE,
)

QUANTITY_REGEX = re.compile(
    rf"""
        ^
        (?:
              (?P<dimension>\d+(?:,\d{{3}})*(?:\.\d+)?\s?{LENGTH_UNIT}\s*[xX]\s*\d+(?:,\d{{3}})*(?:\.\d+)?\s?{LENGTH_UNIT}(?:\s*[xX]\s*\d+(?:,\d{{3}})*(?:\.\d+)?\s?{LENGTH_UNIT})?)
            | \d+(?:,\d{{3}})*(?:\.\d+)?
              (?:
                    (?P<pressure>/(?:\d+(?:,\d{{3}})*(?:\.\d+)?)\s?{PRESSURE_UNIT}(?![A-Za-z]) | \s?{PRESSURE_UNIT}(?![A-Za-z]))
                  | (?P<frequency>/{TIME_UNIT_EXT}(?![A-Za-z]))
                  | (?P<weight>\s?{WEIGHT_UNIT}(?:/{WEIGHT_UNIT})?(?![A-Za-z]))
                  | (?P<volume>\s?{VOLUME_UNIT}(?:/{VOLUME_UNIT})?(?![A-Za-z]))
                  | (?P<time>\s?{TIME_UNIT_EXT}(?:/{TIME_UNIT_EXT})?(?![A-Za-z]))
                  | (?P<lenght>\s?{LENGTH_UNIT}(?:/{LENGTH_UNIT})?(?![A-Za-z]))
                  | (?P<especial>\s?{SPECIAL_UNIT}(?:/{SPECIAL_UNIT})?(?![A-Za-z]))
              )?
        )
        $
    """,
    re.VERBOSE | re.IGNORECASE,
)

def separate_sentences(text: str) -> list[str]:
    """Recebe um texto e retorna uma lista de sentenças."""
    return nltk.tokenize.sent_tokenize(text)

def tokenize(text: str) -> list[str]:
    """Recebe um texto e retorna uma lista de tokens usando o regex personalizado."""
    return [match.group(0) for match in TOKEN_SEP_REGEX.finditer(text)]

def extract_quantities_by_token_index(text: str) -> list[dict]:
    """
    Identifica quantidades passando pelo tokenizer e mapeia pelo índice do token na sentença.
    """
    sentences = separate_sentences(text)
    quantities_found = []

    for sent_idx, sentence in enumerate(sentences):
        tokens = tokenize(sentence)

        for token_idx, token in enumerate(tokens):
            match = QUANTITY_REGEX.match(token)
            if match:
                group_dict = match.groupdict()
                matched_rule = next((name for name, value in group_dict.items() if value is not None), "numero_puro")
                
                quantities_found.append({
                    "sent_index": sent_idx,
                    "sentence": sentence.strip(),
                    "token_index": token_idx,
                    "quantity": token,
                    "matched_rule": matched_rule,
                    "tokens_in_sentence": tokens
                })
    return quantities_found

texto_teste = """A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal pancreatectomy was planned. At the time of the laparoscopic exploration, entrance to the lesser sac was first obtained and, on mobilising the posterior wall of the stomach, the lesion was noted to be tightly adherent to both the stomach and pancreas. The pancreas was mobilised free of the cyst and it was evident that the lesion originated from the stomach. Intraoperative endoscopy was performed and no mucosal abnormalities of the stomach were noted.
On further mobilisation, the cyst was adherent to the coeliac axis and the procedure was converted to an open resection. The stomach was fully mobilised and the gastric duplication cyst was noted to be arising from the posterior wall. It was dissected off the coeliac vessels and an enbloc resection of the cyst along with a portion of the posterior wall of the stomach with a surgical stapler was performed. Repeat oesophagogastroduodenoscopy revealed the presence of an intact staple line on the posterior stomach wall and no evidence of bleeding or leak. The gross margins were uninvolved.
Final pathology revealed a 9.5cm x 4.5cm x 2.0cm cyst with a smooth internal cyst wall consistent with a GDC (Figs 3-5). The patient was discharged home on postoperative day 4, doing well and tolerating a regular diet without any complications. On follow-up, she had complete resolution of her pain and nausea, and a return to regular bowel function and activity."""


resultados = extract_quantities_by_token_index(texto_teste)

for item in resultados:
    print(f"Sentença [{item['sent_index']}]: \"{item['sentence']}\"")
    print(f"  -> Quantidade: \"{item['quantity']}\" | Regra do Regex: **{item['matched_rule'].upper()}** (Token ID: {item['token_index']})")
    print("-" * 75)

Sentença [0]: "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation."
  -> Quantidade: "44" | Regra do Regex: **NUMERO_PURO** (Token ID: 1)
---------------------------------------------------------------------------
Sentença [0]: "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation."
  -> Quantidade: "3" | Regra do Regex: **NUMERO_PURO** (Token ID: 7)
---------------------------------------------------------------------------
Sentença [2]: "She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1)."
  -> Quantidade: "6cm" | Regra do Regex: **LENGHT** (Token ID: 9)
---------------------------------------------------------------------------
Sentença [2]: "She underwent contrast enhanced computed tomography, demonstrating a 6cm cys